In [2]:
import os
os.chdir("C:/TFM/tfm_env/")

import polars as pl
import pandas as pd


In [1]:
# pipeline_financieros_pandas.py
from __future__ import annotations

import os
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, Iterable
from datetime import datetime

import pandas as pd



In [ ]:

# =============== Configuración ===============

@dataclass(frozen=True)
class Config:
    statement_data_dir: Path
    statement_data_dir_dest: Path
    selected_data_dir_dest: Path
    sheets: tuple[str, str] = ("Ratios", "Cash Flow")
    required_cash_flow_fields: tuple[str, ...] = ("% Free Cash Flow Margins",)
    required_ratio_fields: tuple[str, ...] = ("Return on Assets %", 
                                              "Return on Invested Capital %", 
                                            "Return On Equity %", "Normalized ROIC %", 
                                            "Gross Profit Margin %", "EBITDA Margin %", 
                                            "Net Income Margin %", 
                                            "Normalized Net Income Margin %", 
                                            "Current Ratio", "Total Debt / Equity")
    min_periods: int = 16
    min_latest_year: int = 2024


In [15]:
config = Config(statement_data_dir="data/01_raw/01.3 - HistoricoEstadosFinancieros/",
                statement_data_dir_dest="data/02_intermediate/02.1 - HistoricoEstadosFinancieros/",
                selected_data_dir_dest="data/02_intermediate/02.2 - SeleccionHistoricoEstadosFinancieros/"
                )

In [ ]:

# =============== Utilidades genéricas ===============

def ensure_dirs(*paths: Path) -> None:
    for p in paths:
        p.mkdir(parents=True, exist_ok=True)


def sector_dirs(root: Path) -> Iterable[Path]:
    # Solo subdirectorios (cada sector)
    for p in sorted(root.iterdir()):
        if p.is_dir():
            yield p


def excel_paths(sector_dir: Path) -> Iterable[Path]:
    for p in sorted(sector_dir.iterdir()):
        if p.suffix.lower() in {".xlsx", ".xls"} and p.is_file():
            yield p


def parse_date_cols(cols: list[str], title_col: str) -> list[datetime]:
    # Quita el título y "LTM"; convierte dd/mm/yy -> datetime
    candidates = [c for c in cols if c not in {title_col, "LTM"}]
    dates = pd.to_datetime(candidates, format="%d/%m/%y", errors="coerce")
    return [d.to_pydatetime() for d in dates if pd.notna(d)]


def date_guard(df: pd.DataFrame, title_col: str, *,
               min_periods: int, min_latest_year: int) -> bool:
    dates = parse_date_cols(df.columns.tolist(), title_col)
    if len(dates) < min_periods:
        return False
    latest = max(dates).year if dates else 0
    return latest >= min_latest_year


def set_index_and_pick_date_cols(df: pd.DataFrame, title_col: str) -> pd.DataFrame:
    # Deja fuera "LTM" y mantiene solo columnas-fecha; índice = fila de títulos
    out = df.set_index(title_col)
    date_cols = [c for c in out.columns if c != "LTM"]
    return out[date_cols]


# =============== Lectura y validación por archivo ===============

def read_both_sheets(path: Path, cfg: Config) -> Dict[str, pd.DataFrame]:
    # Lee ambas hojas de una sola pasada (dict de DataFrames)
    sheets_map = pd.read_excel(
        path,
        sheet_name=list(cfg.sheets),  # ["Ratios", "Cash Flow"]
        engine=None  # deja que pandas elija
    )
    # Normaliza claves a str exactos de cfg.sheets
    return {name: sheets_map[name] for name in cfg.sheets}


def validate_file(
    ratios_df: pd.DataFrame,
    cash_df: pd.DataFrame,
    cfg: Config,
) -> bool:
    title_idx_ratio = f"{cfg.sheets[0]} | TIKR.com"
    title_idx_cash = f"{cfg.sheets[1]} | TIKR.com"

    # 1) Guard de fechas y número de periodos
    if not date_guard(ratios_df, title_idx_ratio,
                      min_periods=cfg.min_periods, min_latest_year=cfg.min_latest_year):
        return False
    if not date_guard(cash_df, title_idx_cash,
                      min_periods=cfg.min_periods, min_latest_year=cfg.min_latest_year):
        return False

    # 2) Validaciones de contenido (usa tus funciones existentes)
    if not is_valid_file_cash_flow(cash_df, list(cfg.required_cash_flow_rows)):
        return False
    if not is_valid_file_ratio(ratios_df, ratios_list):
        return False

    return True


# =============== Transformaciones (pipeline por archivo) ===============

def cash_to_fcf_series(cash_df: pd.DataFrame, cfg: Config) -> pd.Series:
    title_idx_cash = f"{cfg.sheets[1]} | TIKR.com"
    # (1) fijar índice y quedarnos con columnas-fecha
    # (2) transponer -> filas=fechas; (3) seleccionar la fila "% Free Cash Flow Margins" como Serie
    return (
        cash_df
        .pipe(set_index_and_pick_date_cols, title_idx_cash)  # df[dates], index=metricas
        .transpose()[cfg.required_cash_flow_rows[0]]         # Serie por fecha
    )


def ratios_to_frame(ratios_df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    title_idx_ratio = f"{cfg.sheets[0]} | TIKR.com"
    return (
        ratios_df
        .pipe(set_index_and_pick_date_cols, title_idx_ratio)  # df[dates], index=metricas
        .transpose()[ratios_list]                             # df por fecha con columnas de ratios
    )


def merge_ratios_and_fcf(r_df: pd.DataFrame, fcf_s: pd.Series) -> pd.DataFrame:
    # Alinea por índice (fechas)
    return pd.concat([r_df, fcf_s.rename("% Free Cash Flow Margins")], axis=1)


def add_metadata(df: pd.DataFrame, *, ticker: str, sector: str) -> pd.DataFrame:
    return (
        df
        .assign(ticker=ticker, sector=sector)
        .rename_axis(index="date")  # opcional: nombra el índice (antes 'id' en tu CSV)
        .reset_index()              # si prefieres fecha como columna y 'id' autoincremental en CSV
    )


def process_one_file(path: Path, sector_name: str, cfg: Config) -> Optional[pd.DataFrame]:
    try:
        sheets = read_both_sheets(path, cfg)
        ratios_df = sheets[cfg.sheets[0]]
        cash_df   = sheets[cfg.sheets[1]]

        if not validate_file(ratios_df, cash_df, cfg):
            return None

        fcf_s = cash_df.pipe(cash_to_fcf_series, cfg)
        r_df  = ratios_df.pipe(ratios_to_frame, cfg)
        merged = merge_ratios_and_fcf(r_df, fcf_s)

        ticker = extract_tikr(path.name)  # tu función existente
        out = add_metadata(merged, ticker=ticker, sector=sector_name)
        return out

    except Exception as e:
        print(f"[WARN] Saltando {path.name}: {e}")
        return None


# =============== Escritura de salidas ===============

def write_per_sheet_csvs(path: Path, sheets_map: Dict[str, pd.DataFrame], cfg: Config) -> None:
    base = cfg.statement_data_dir_dest / path.parent.name  # sector
    ensure_dirs(base)
    stem = path.stem
    for sheet, df in sheets_map.items():
        out = base / f"{stem}-{sheet}.csv"
        df.to_csv(out, index=False)
        print(f"Guardado: {out}")


def write_per_ticker_csv(df: pd.DataFrame, ticker: str, cfg: Config) -> None:
    ensure_dirs(cfg.selected_data_dir_dest)
    out = cfg.selected_data_dir_dest / f"{ticker}-data.csv"
    df.to_csv(out, index=False)


# =============== Orquestador ===============

def run_pipeline(cfg: Config) -> pd.DataFrame:
    ensure_dirs(cfg.statement_data_dir_dest, cfg.selected_data_dir_dest)

    all_rows: list[pd.DataFrame] = []

    for sdir in sector_dirs(cfg.statement_data_dir):
        sector_name = sdir.name  # e.g., "ConsumoDiscrecional"
        for xls in excel_paths(sdir):

            # Lee una vez para guardar "sheets" CSV igual que tu script original
            try:
                sheets_map = read_both_sheets(xls, cfg)
                write_per_sheet_csvs(xls, sheets_map, cfg)
            except Exception as e:
                print(f"[WARN] No se pudieron exportar sheets de {xls.name}: {e}")

            # Procesa el archivo con el pipeline por archivo
            df = process_one_file(xls, sector_name, cfg)
            if df is None:
                continue

            # CSV por ticker
            try:
                ticker = df["ticker"].iloc[0]
                write_per_ticker_csv(df, ticker, cfg)
            except Exception as e:
                print(f"[WARN] No se pudo escribir CSV de ticker para {xls.name}: {e}")

            all_rows.append(df)

    if not all_rows:
        print("[INFO] No hay datos válidos.")
        return pd.DataFrame()

    all_data = pd.concat(all_rows, axis=0, ignore_index=True)

    # Escribe el unificado una sola vez (ya no hace falta manejar header=True/False)
    unified = Path("data/02_intermediate/02.2.1 SeleccionHistoricoEstadosFinancierosUnificado")
    unified.mkdir(parents=True, exist_ok=True)
    all_data.to_csv(unified / "AllData.csv", index=False)

    return all_data


# =============== Ejemplo de uso ===============
if __name__ == "__main__":
    cfg = Config(
        statement_data_dir=Path("data/01_raw/01.3 - HistoricoEstadosFinancieros/"),
        statement_data_dir_dest=Path("data/02_intermediate/02.1 - HistoricoEstadosFinancieros/"),
        selected_data_dir_dest=Path("data/02_intermediate/02.2 - SeleccionHistoricoEstadosFinancieros/")
    )

    # ratios_list, is_valid_file_cash_flow, is_valid_file_ratio, extract_tikr deben existir en tu entorno
    all_df = run_pipeline(cfg)
    print("Filas totales:", len(all_df))
